# Defining the subevent matching algorithm

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import uproot
import sys
import seaborn as sns
from tqdm import tqdm
import networkx as nx
import matplotlib.cm as cm

import atlasify as atl
from particle import Particle
atl.ATLAS = "ColliderML"

sys.path.append("/global/cfs/cdirs/m4958/usr/danieltm/ColliderML/software/OtherLibraries/pyedm4hep")
from pyedm4hep import EDM4hepEvent, EDM4hepEventBatch

## Eventwise

In [2]:
edm_input_file = "/pscratch/sd/d/danieltm/ColliderML/simulation/full_pileup_pilot/ttbar/v2/runs/0/edm4hep.root"
event = EDM4hepEvent(edm_input_file, event_index=0)

Loading event 0 from /pscratch/sd/d/danieltm/ColliderML/simulation/full_pileup_pilot/ttbar/v2/runs/0/edm4hep.root...
  Loaded 880327 particles.
  Loaded 240622 tracker hits.
  Loaded 1243953 calo hits and 6038426 contributions.


In [3]:
hits = event.get_tracker_hits_df()

In [4]:
calo_hits = event.get_calo_contributions_df()

In [5]:
particles = event.get_particles_df()

In [6]:
%%time

for i in range(10):
    event = EDM4hepEvent(edm_input_file, event_index=i)
    hits = event.get_tracker_hits_df()

CPU times: user 4 μs, sys: 1 μs, total: 5 μs
Wall time: 10.7 μs
Loading event 0 from /pscratch/sd/d/danieltm/ColliderML/simulation/full_pileup_pilot/ttbar/v2/runs/0/edm4hep.root...
  Loaded 880327 particles.
  Loaded 240622 tracker hits.
  Loaded 1243953 calo hits and 6038426 contributions.
Loading event 1 from /pscratch/sd/d/danieltm/ColliderML/simulation/full_pileup_pilot/ttbar/v2/runs/0/edm4hep.root...
  Loaded 948682 particles.
  Loaded 268168 tracker hits.
  Loaded 1330005 calo hits and 6470534 contributions.
Loading event 2 from /pscratch/sd/d/danieltm/ColliderML/simulation/full_pileup_pilot/ttbar/v2/runs/0/edm4hep.root...
  Loaded 949723 particles.
  Loaded 252857 tracker hits.
  Loaded 1329280 calo hits and 6537286 contributions.
Loading event 3 from /pscratch/sd/d/danieltm/ColliderML/simulation/full_pileup_pilot/ttbar/v2/runs/0/edm4hep.root...
  Loaded 946346 particles.
  Loaded 251185 tracker hits.
  Loaded 1309217 calo hits and 6497732 contributions.
Loading event 4 from /ps

## Batch loading

In [10]:
edm_input_file = "/pscratch/sd/d/danieltm/ColliderML/simulation/full_pileup_pilot/ttbar/v2/runs/0/edm4hep.root"
batch = EDM4hepEventBatch(edm_input_file, events=slice(0, 100))

In [11]:
hits_batch = batch.get_tracker_hits_df()

In [12]:
hits_batch.head()

,event_id,subentry,cellID,time,pathLength,quality,x,y,z,px,...,pz,EDep,detector,particle_id,r,R,phi,theta,eta,pt
0,0,0,256188893628166,-4.977191,0.163255,0,-31.831104,4.829101,-489.650446,-0.137585,...,0.047277,0.000059,PixelBarrelReadout,84489,32.195332,490.707753,2.991031,3.075935,-3.416097,0.235782
1,0,1,256802604189702,-4.969477,0.259380,0,-32.956036,6.378156,-489.269836,-0.139093,...,0.046564,0.000079,PixelBarrelReadout,84489,33.567561,490.419977,2.950421,3.073093,-3.373676,0.235751
2,0,2,267388356398358,-4.817920,0.134571,0,-57.763868,35.774230,-481.510196,-0.163330,...,0.047070,0.000053,PixelBarrelReadout,84489,67.944536,486.280299,2.587089,3.001411,-2.656324,0.233511
3,0,3,280237556372774,-4.633086,0.128650,0,-93.217879,66.303405,-472.187467,-0.189751,...,0.046297,0.000044,PixelBarrelReadout,84489,114.392808,485.846394,2.523341,2.903911,-2.125245,0.232564
4,0,4,14086720987190,-4.410011,0.127594,0,-142.769602,93.210803,-461.073716,-0.214661,...,0.045286,0.000041,PixelBarrelReadout,84489,170.503410,491.589651,2.563199,2.787392,-1.720506,0.231859


In [13]:
hits_batch.shape

(24914797, 21)

## Size Testing

Currently 2.2GB

In [24]:
import h5py

h5_file = h5py.File("/pscratch/sd/d/danieltm/ColliderML/simulation/full_pileup_pilot/ttbar/v2/reco/tracker_hits/full_pileup_pilot.ttbar.v2.reco.tracker_hits.events0-99.h5", "r")

In [23]:
import pandas as pd
pd.DataFrame(h5_file["events"]["event_0"]["measurements"][:])

,x,y,z,volume_id,layer_id,surface_id,true_x,true_y,true_z,time,px,py,pz,particle_id,cell_id,detector,e_dep,path_length
0,85.607491,3.613425,-1515.599976,16,4,1,85.631775,3.625161,-1515.599976,35.864868,-0.127980,0.013338,-0.021521,703493,67494740071,1,0.000370,0.821243
1,92.881134,-3.039804,-1515.599976,16,4,1,92.851669,-3.027657,-1515.599976,13.531095,0.165640,-0.054582,-2.974995,67930,1023410279,1,0.000036,0.125214
2,66.319473,-7.858758,-1515.599976,16,4,1,66.317802,-7.857886,-1515.599976,9.396918,0.268940,0.010043,-6.876966,74845,2634023015,1,0.000029,0.125096
3,65.872543,-3.477791,-1515.599976,16,4,1,65.890350,-3.451361,-1515.599976,8655.912109,-0.011574,-0.037811,-0.014304,669642,1157628007,1,0.000116,0.369049
4,92.184578,-4.563632,-1515.599976,16,4,1,92.188545,-4.542456,-1515.599976,6.495801,-0.001369,0.025355,-0.071781,375396,1526726759,1,0.000101,0.132867
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
240609,406.731476,903.742432,3009.500000,30,12,192,421.881561,942.290344,3009.500000,7.946029,0.693225,0.425183,2.162005,77697,858994544990,5,0.000081,0.267095
240610,325.683228,935.429993,3009.500000,30,12,192,306.645935,886.782227,3009.500000,11.627073,0.005575,0.012120,-0.007825,343955,15955804590430,5,0.000144,0.522783
240611,353.611938,924.510681,3009.500000,30,12,192,331.738159,868.459473,3009.500000,10.063175,0.001385,0.003332,-0.001101,573373,16819093016926,5,0.000224,0.663010
240612,435.766296,892.390686,3009.500000,30,12,192,446.899231,921.189087,3009.500000,7.529224,-0.001016,-0.001745,0.001744,759736,1748052775262,5,0.000116,0.420584


In [25]:
import pandas as pd
pd.DataFrame(h5_file["events"]["event_0"]["measurements"][:])

,x,y,z,volume_id,layer_id,surface_id,true_x,true_y,true_z,time,px,py,pz,particle_id,cell_id,detector,e_dep,path_length
0,85.607491,3.613425,-1515.599976,16,4,1,85.631775,3.625161,-1515.599976,35.864868,-0.127980,0.013338,-0.021521,703493,67494740071,1,0.000370,0.821243
1,92.881134,-3.039804,-1515.599976,16,4,1,92.851669,-3.027657,-1515.599976,13.531095,0.165640,-0.054582,-2.974995,67930,1023410279,1,0.000036,0.125214
2,66.319473,-7.858758,-1515.599976,16,4,1,66.317802,-7.857886,-1515.599976,9.396918,0.268940,0.010043,-6.876966,74845,2634023015,1,0.000029,0.125096
3,65.872543,-3.477791,-1515.599976,16,4,1,65.890350,-3.451361,-1515.599976,8655.912109,-0.011574,-0.037811,-0.014304,669642,1157628007,1,0.000116,0.369049
4,92.184578,-4.563632,-1515.599976,16,4,1,92.188545,-4.542456,-1515.599976,6.495801,-0.001369,0.025355,-0.071781,375396,1526726759,1,0.000101,0.132867
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
240609,406.731476,903.742432,3009.500000,30,12,192,421.881561,942.290344,3009.500000,7.946029,0.693225,0.425183,2.162005,77697,858994544990,5,0.000081,0.267095
240610,325.683228,935.429993,3009.500000,30,12,192,306.645935,886.782227,3009.500000,11.627073,0.005575,0.012120,-0.007825,343955,15955804590430,5,0.000144,0.522783
240611,353.611938,924.510681,3009.500000,30,12,192,331.738159,868.459473,3009.500000,10.063175,0.001385,0.003332,-0.001101,573373,16819093016926,5,0.000224,0.663010
240612,435.766296,892.390686,3009.500000,30,12,192,446.899231,921.189087,3009.500000,7.529224,-0.001016,-0.001745,0.001744,759736,1748052775262,5,0.000116,0.420584
